# 第1章 LangChain 基礎

LangChain経由で Amazon Bedrock のLLMを呼び出し、`Prompt → Model → Parser` のチェーン(LCEL)を体験します。

**所要時間**: 約20分

## 0. 準備 - 環境変数の読み込み

`.env` から `AWS_PROFILE` / `AWS_REGION` / モデルIDを取り込みます。

**`load_dotenv()` の役割**: カレントディレクトリにある `.env` ファイルを読み込み、その中の値を Python の環境変数(`os.environ`)に追加します。
これにより、後で `boto3`(Bedrockへの通信ライブラリ)が `AWS_PROFILE` を自動的に拾い、SSO認証情報を使えるようになります。

In [ ]:
import os
from dotenv import load_dotenv

# .env ファイルを読み込み、その中の KEY=VALUE を os.environ に注入する
# 戻り値は読み込めたかの bool だが、ここでは気にしない
load_dotenv()

# .env で定義したキーを取得。存在しなければ KeyError(ハンズオンでは事前に必ず設定する)
AWS_REGION = os.environ["AWS_REGION"]              # 呼び出し先のAWSリージョン (例: ap-northeast-1)
CHAT_MODEL_ID = os.environ["BEDROCK_CHAT_MODEL_ID"]  # 使用するチャットモデルのID

# 確認のため表示
print("region:", AWS_REGION)
print("chat model:", CHAT_MODEL_ID)
print("profile:", os.environ.get("AWS_PROFILE"))  # SSO等で使用するプロファイル名

## 1. Hello Bedrock

`ChatBedrockConverse` は Bedrock の **Converse API** を使うチャットモデルクラスです。
ツール呼び出しなどがモデル横断で扱えるため、本教材では基本これを使います。

**ポイント**
- `ChatBedrockConverse` は LangChain における「チャット用LLM」の抽象化された型。OpenAIなら `ChatOpenAI`、Anthropic直なら `ChatAnthropic` に相当します。
- `.invoke(入力)` でモデルに1回問い合わせます。これは LangChain の全LLM共通のメソッド名です。
- 戻り値は `AIMessage` という型のオブジェクト。生のテキストは `.content` で取り出します。

In [ ]:
# LangChain の AWS 用パッケージから、Converse API ベースのチャットモデルクラスをインポート
from langchain_aws import ChatBedrockConverse

# LLM クライアントを初期化する
# - model: 呼び出すモデルID
# - region_name: Bedrock のエンドポイントが存在するリージョン
# - temperature: 出力のランダム性。0 だと毎回ほぼ同じ応答、1 に近づくほど多様な応答
llm = ChatBedrockConverse(
    model=CHAT_MODEL_ID,
    region_name=AWS_REGION,
    temperature=0,
)

# .invoke() に文字列を渡すと、内部で human メッセージ1件として扱われ Bedrock に送信される
# 戻り値は AIMessage オブジェクト
response = llm.invoke("こんにちは。あなたは何ができますか? 30字程度で答えてください。")

# .content には生成されたテキストが入っている
print(response.content)

応答テキストが返ってくれば、Bedrockへの接続OKです。

`response` は `AIMessage` オブジェクト。`.content` がテキスト本体です。

## 2. PromptTemplate でプロンプトを構造化

プロンプトをハードコードせず、テンプレ化して変数で差し込めるようにします。

LLM への入力は基本的に「メッセージのリスト」で、各メッセージには **役割(role)** があります:
- `system`: モデルへの指示(役割やルール)
- `human`: ユーザーからの入力
- `ai`: 過去にモデルが返した応答(会話履歴用)

`ChatPromptTemplate.from_messages([...])` は、このメッセージ列のテンプレートを定義します。
`{変数名}` という形でプレースホルダを埋め込め、後で `format_messages(変数名=値)` で実体化できます。

In [ ]:
# プロンプトテンプレートクラスをインポート
from langchain_core.prompts import ChatPromptTemplate

# (役割, テンプレート文字列) のタプルを並べることで、メッセージ列のテンプレートを定義する
# {role} や {question} の部分は後から差し込めるプレースホルダ
prompt = ChatPromptTemplate.from_messages([
    ("system", "あなたは{role}です。簡潔に日本語で答えてください。"),
    ("human", "{question}"),
])

# .format_messages(...) でプレースホルダを実値に置き換え、メッセージのリストを返す
# (ここではまだ LLM を呼んでいない。プロンプトの中身確認用)
messages = prompt.format_messages(role="プログラミング講師", question="LangChainを一言で説明して")

# 出来上がったメッセージ列を表示。type は "system"/"human" などのロールが入る
for m in messages:
    print(f"[{m.type}] {m.content}")

In [ ]:
# 先ほど作ったメッセージ列(system + human の2件)をそのまま LLM に投げる
# .invoke() は文字列だけでなく、メッセージのリストも受け取れる
response = llm.invoke(messages)
print(response.content)

## 3. LCEL: パイプでチェーンを組む

LangChain Expression Language (LCEL) は **コンポーネントを `|` で繋いでパイプライン化** する書き方です。

```
prompt  |  llm  |  StrOutputParser
   ↑        ↑          ↑
 入力整形  推論     文字列に整形
```

**仕組み**
- LangChain の主要コンポーネント(prompt / llm / parser など)は全て `Runnable` というインタフェースを実装しています。
- `Runnable` 同士は `|` 演算子で繋げて、新しい `Runnable`(=チェーン)を作れます。
- 出来上がったチェーンは、また `.invoke()` / `.stream()` / `.batch()` などで呼び出せます。

**各部品の役割**
- `prompt`: 入力 dict (`{"role": ..., "question": ...}`) を、フォーマット済みメッセージ列に変換
- `llm`: メッセージ列を Bedrock に送り、`AIMessage` を返す
- `StrOutputParser`: `AIMessage` から `.content` を取り出し、ただの文字列にする

これにより、チェーン全体は **「dict → 文字列」** という単純な関数のように扱えます。

In [ ]:
# AIMessage オブジェクトから文字列を抽出する Output Parser をインポート
from langchain_core.output_parsers import StrOutputParser

# LCEL: パイプ演算子 | で 3 つのコンポーネントを連結し、1 つのチェーンを作る
# データの流れ: 入力dict → prompt(メッセージ列化) → llm(推論) → parser(文字列化)
chain = prompt | llm | StrOutputParser()

# .invoke() に渡す dict のキーは、prompt の {変数名} と一致させる必要がある
answer = chain.invoke({
    "role": "料理研究家",
    "question": "カレーを美味しく作る3つのコツを教えて",
})

# answer は AIMessage ではなく、すでに文字列(StrOutputParser が変換済み)
print(answer)

## 4. ストリーミングで応答を受け取る

LCELで組んだチェーンは、そのまま `.stream()` でトークン単位の応答が得られます。
UIに逐次表示したい時などに便利です。

**`.invoke()` と `.stream()` の違い**
| メソッド | 戻り値 | 用途 |
|---|---|---|
| `.invoke(input)` | 最終結果(文字列など) | バッチ処理、結果がまとまってから次に進みたい時 |
| `.stream(input)` | チャンクが順次届く iterator | チャットUIに逐次表示したい時 |

`StrOutputParser` を最後に挟んでいるので、ストリーミングでも各チャンクは文字列として届きます。

In [ ]:
# .stream() は generator を返す。for ループで回すと、生成途中のテキストが少しずつ届く
# 終わるまでブロックする .invoke() とは異なり、最初のトークンが届いた時点でループが始まる
for chunk in chain.stream({
    "role": "歴史の先生",
    "question": "日本の応仁の乱について100字くらいで説明して",
}):
    # 各 chunk は文字列の断片。end="" と flush=True で改行せずリアルタイム表示
    print(chunk, end="", flush=True)

## まとめ

- `ChatBedrockConverse` で Bedrock のチャットモデルを呼べる
- `ChatPromptTemplate` で入力を構造化できる
- `prompt | llm | parser` のLCELで、推論パイプラインを宣言的に組める
- 同じチェーンが `.invoke()` / `.stream()` 両対応

次は [02 RAG](../02_rag/rag.ipynb) で、外部文書を参照する仕組みを作ります。